# Collocation runtime profiling: why is the exact Hessian slow on GPU?

The 2x3 grid in `gpu_benchmark_collocation.ipynb` left one result unexplained:

| variant | Amdahl ceiling (claimed) | measured GPU speedup |
|---|---|---|
| Gauss-Newton | 3.6x | **4.06x** |
| exact Hessian | 21.8x | **1.71x** |

Gauss-Newton **exceeded its own stated ceiling**, which is impossible. So the cost split those
ceilings come from is measuring something the benchmark arms do not do, and the size of the
exact-Hessian anomaly is unknown -- it may be large, small, or absent.

One hypothesis is already dead: **fp64 throughput**. Consumer cards run fp64 at 1/32-1/64 of
fp32, which would have explained it neatly -- but the 1.71x was measured on an **A100**, where
fp64 runs at 1/2. That is not the cause.

This notebook does not assume an answer. It measures three specific things:

**Part A -- are the ceilings even real?** `profile_split` in the benchmark runs `maxiter=30`
while the arms run 300, so fixed per-solve costs (composition, first-call warm-up, transcription
setup) are amortised over 10x fewer iterations and inflate the apparent non-torch fraction.
Sweeping `maxiter` shows whether the split converges to something different, and whether the
resulting ceiling is consistent with the measured speedup.

**Part B -- is the timing itself valid on CUDA?** Nothing in the estimator or the benchmark
calls `torch.cuda.synchronize()`. CUDA kernels are asynchronous, so a `perf_counter` around a
torch callback can measure launch time rather than execution time. It may well be fine --
each callback returns host arrays to CasADi, which forces an implicit sync -- but "probably
fine" is not a measurement. This compares explicit-sync against no-sync timing.

**Part C -- the chunking.** `_hchunk = _deriv_chunk // 2` halves the Hessian vmap
*unconditionally*. On this problem the entire Hessian needs 0.30 GB against a 2 GB budget, so
the Jacobian runs as ONE vmap while the Hessian is split into TWO sequential passes for no
memory reason. Note `TWIN4BUILD_DERIV_BYTES` **cannot** fix this: `_deriv_chunk` is capped at
the segment count, not the budget, so no budget lifts `_hchunk` above `n_seg/2`. The divisor is
therefore exposed as `TWIN4BUILD_HESS_CHUNK_DIV` (default 2) purely so its cost can be measured
before anyone changes the default.

**Setup**: Runtime > Change runtime type > **GPU** (A100 preferred; fp64 matters), then Run all.
Roughly 25-40 minutes. Without a GPU the CPU halves still run and Part A is still meaningful.

In [ ]:
# --- Setup (Colab-aware) ---------------------------------------------------
# Needs TWIN4BUILD_HESS_CHUNK_DIV, added alongside this notebook.
TWIN4BUILD_REF = "fix/issue-damper-ventilation-identifiability"

try:
    import twin4build as tb
except ImportError:
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"],
        check=True,
    )
    import twin4build as tb

# Fail loudly now rather than 30 minutes in.
import inspect

import twin4build.estimator._transcription as _tr

_src = inspect.getsource(_tr)
_missing = [n for n in ("TWIN4BUILD_HESS_CHUNK_DIV", "exact_hessian",
                        "boundary_state_init") if n not in _src]
if _missing:
    raise RuntimeError(
        "The installed twin4build is missing: " + ", ".join(_missing)
        + f"""
Installed at: {tb.__file__}
Install a ref that has these, then restart the runtime:
    pip install -q --force-reinstall --no-deps git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}
    (Colab: Runtime > Restart session, then re-run this cell)"""
    )

import functools
import os
import time

import numpy as np
import pandas as pd
import torch

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])


def hardware():
    import platform
    import re

    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as fh:
            for line in fh:
                if line.lower().startswith("model name"):
                    cpu = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass
    ram = None
    try:
        with open("/proc/meminfo") as fh:
            ram = int(re.search(r"[0-9]+", fh.readline()).group()) / 1e6
    except OSError:
        pass
    gpu = fp64_ratio = None
    if torch.cuda.is_available():
        pr = torch.cuda.get_device_properties(0)
        gpu = f"{pr.name} ({pr.total_memory / 1e9:.0f} GB, sm_{pr.major}{pr.minor})"
        # Datacenter parts (A100/V100, sm_70/80/90) run fp64 at 1/2 of fp32;
        # consumer parts at 1/32-1/64.  Recorded because it was the first
        # hypothesis for the exact-Hessian result and it needs to stay visible.
        fp64_ratio = "1/2 (datacenter)" if pr.major in (7, 8, 9) and pr.minor == 0 \
            else "1/32-1/64 (consumer) -- suspect"
    return {"cpu": cpu, "cores": os.cpu_count(),
            "torch_threads": torch.get_num_threads(),
            "ram_gb": round(ram, 1) if ram else None,
            "gpu": gpu, "fp64": fp64_ratio, "torch": torch.__version__}


HW = hardware()
for k, v in HW.items():
    print(f"{k:14s} {v}")
if not torch.cuda.is_available():
    print("\nNO GPU -- Part A still works; the GPU comparisons will be skipped.")

## The model

Pinned copy of `gpu_benchmark_collocation.ipynb`'s problem (28 parameter groups, 4 sensors,
5-day window at 20-minute steps), so timings are comparable to that notebook's grid.

`UA` takes its start value from `initialize_UA` (`x0=None`). Hardcoding it measurably changes
the optimum reached -- pooled 44.71 vs 33.89 on one machine -- so it is left physics-derived
here.

In [ ]:
import datetime
import importlib.util as _ilu
import pathlib

from dateutil import tz

import twin4build.examples.utils as utils
from twin4build.utils.rgetattr import rgetattr

# `twin4build/examples/full_workflow_example/` (packaged CSVs) is a PACKAGE that
# shadows the module `full_workflow_example.py`, so load the module by path.
import twin4build.examples as _ex_pkg

_p = pathlib.Path(_ex_pkg.__file__).parent / "full_workflow_example.py"
_spec = _ilu.spec_from_file_location("_fwe_runtime", _p)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fcn = _mod.fcn

STEP = 1200
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]
N_WARMUP = 20


def build_model(device="cpu", dtype=torch.float64, tag="prof"):
    m = tb.Model(id=f"{tag}_{device}")
    m.load(semantic_model_filename=utils.get_path(
        ["estimator_example", "one_room_example_model.xlsm"]), fcn=fcn)
    m.to(device, dtype)
    return m


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    return [(model.components["office_valve_position_sensor"], 0.05 / 2),
            (model.components["office_temperature_sensor"], 0.1 / 2),
            (model.components["office_damper_position_sensor"], 0.05 / 2),
            (model.components["office_co2_sensor"], 30 / 2)]


COLLOC_OPTS = {"boundary_state_init": "rollout", "early_stopping": False}
print("model builders ready")

## Part A -- are the Amdahl ceilings real?

A GPU can only accelerate the **torch** callbacks (objective, gradient, constraints, constraint
Jacobian, Hessian). IPOPT's sparse KKT factorisation stays on the CPU. So

    ceiling = 1 / (1 - torch_fraction)

is a hard upper bound on any GPU speedup. **Gauss-Newton measured 4.06x against a stated 3.6x
ceiling, which cannot happen** -- so the published split is wrong, and the first job is to find
out how.

The suspected cause is the iteration count. Fixed per-solve costs -- graph composition, first-call
warm-up, transcription setup -- are paid once and land inside the *non-torch* bucket. At
`maxiter=30` they are spread over 30 iterations; at 300, over 300. If that is the explanation,
`torch_frac` will **rise** with `maxiter` and the ceiling with it.

The profiler below also records `setup_s` (everything before the first callback) separately, so
the fixed cost is visible rather than inferred.

In [ ]:
import twin4build.estimator._casadi_ipopt as _ipopt


def profile_split(device, exact, maxiter=30, sync=False, hess_div=None, dtype=torch.float64):
    """Time each torch callback against total solve time.

    sync=True inserts torch.cuda.synchronize() inside every timed callback, so
    asynchronous kernel execution is charged to the callback that launched it
    rather than to whatever later call happens to force a sync.
    """
    T = {k: 0.0 for k in ("obj", "grad", "g", "gjac", "hess")}
    first_call = {}
    do_sync = sync and device == "cuda"

    def timed(fn, key):
        if fn is None:
            return None

        @functools.wraps(fn)   # keep the signature visible: solve_ipopt_constrained
        def w(*a, **kw):       # probes hess_vals arity to decide on lam_g
            t0 = time.perf_counter()
            try:
                return fn(*a, **kw)
            finally:
                if do_sync:
                    torch.cuda.synchronize()
                T[key] += time.perf_counter() - t0
                first_call.setdefault("t", t0)

        return w

    env_prev = os.environ.get("TWIN4BUILD_HESS_CHUNK_DIV")
    if hess_div is not None:
        os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = str(hess_div)

    model = build_model(device, dtype, tag="split")
    est = tb.Estimator(tb.Simulator(model))
    orig = _ipopt.solve_ipopt_constrained
    box = {}

    def wrapped(x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jr, jc,
                options=None, *, hess_vals=None, **kw):
        t0 = time.perf_counter()
        try:
            return orig(x0, lb, ub, timed(fun, "obj"), timed(grad, "grad"), n_g,
                        timed(g_fun, "g"), timed(g_jac_vals, "gjac"), jr, jc,
                        options, hess_vals=timed(hess_vals, "hess"), **kw)
        finally:
            box["total"] = time.perf_counter() - t0
            box["t0"] = t0

    _ipopt.solve_ipopt_constrained = wrapped
    t_wall = time.perf_counter()
    try:
        opts = dict(COLLOC_OPTS)
        opts.update({"maxiter": maxiter, "exact_hessian": exact})
        est.estimate(START, END, STEP, build_parameters(model),
                     build_measurements(model), n_warmup=N_WARMUP,
                     method=("casadi", "ipopt", "ad", "collocation"), options=opts)
    finally:
        _ipopt.solve_ipopt_constrained = orig
        if env_prev is None:
            os.environ.pop("TWIN4BUILD_HESS_CHUNK_DIV", None)
        else:
            os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = env_prev

    total = box["total"]
    torch_s = sum(T.values())
    frac = torch_s / total
    return {
        "device": device,
        "variant": "exact" if exact else "gauss-newton",
        "maxiter": maxiter,
        "sync": sync,
        "hess_div": hess_div if hess_div is not None else 2,
        "solve_s": total,
        # Everything before the first callback: composition, warm-up, transcription.
        "setup_s": first_call.get("t", box["t0"]) - box["t0"],
        "torch_frac": frac,
        "ipopt_frac": 1 - frac,
        "s_per_iter_ipopt": (total - torch_s) / maxiter,
        "amdahl_ceiling": 1.0 / max(1e-9, 1 - frac),
        **{f"{k}_s": v for k, v in T.items()},
    }


# Sweep maxiter: if fixed costs are the contamination, torch_frac rises with it.
ITERS = [30, 100, 300]
rows_a = []
for dev in DEVICES:
    for exact in (False, True):
        for mi in ITERS:
            print(f"  {dev} | {'exact' if exact else 'gauss-newton'} | maxiter={mi}",
                  flush=True)
            try:
                rows_a.append(profile_split(dev, exact, maxiter=mi))
            except Exception as exc:
                print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)

split = pd.DataFrame(rows_a)
split[["device", "variant", "maxiter", "solve_s", "setup_s", "torch_frac",
       "amdahl_ceiling", "s_per_iter_ipopt"]]

### Is the ceiling consistent with the measured speedup?

A ceiling below an observed speedup is a **self-refuting measurement**. The check below compares
each variant's CPU-derived ceiling against the GPU speedup actually measured at the same
`maxiter`, and says plainly whether the numbers can both be true.

In [ ]:
if "cuda" in DEVICES and len(split):
    checks = []
    for variant in ("gauss-newton", "exact"):
        for mi in ITERS:
            c = split[(split.variant == variant) & (split.maxiter == mi) &
                      (split.device == "cpu")]
            g = split[(split.variant == variant) & (split.maxiter == mi) &
                      (split.device == "cuda")]
            if not len(c) or not len(g):
                continue
            ceiling = float(c.amdahl_ceiling.iloc[0])
            speedup = float(c.solve_s.iloc[0]) / float(g.solve_s.iloc[0])
            checks.append({
                "variant": variant, "maxiter": mi,
                "cpu_torch_frac": float(c.torch_frac.iloc[0]),
                "ceiling": ceiling, "measured_speedup": speedup,
                "consistent": speedup <= ceiling * 1.05,   # 5% measurement slack
            })
    chk = pd.DataFrame(checks)
    display(chk)
    bad = chk[~chk.consistent]
    if len(bad):
        print("*** IMPOSSIBLE: measured speedup exceeds the Amdahl ceiling ***")
        print("    The cost split is not measuring what the arms do.")
        for _, r in bad.iterrows():
            print(f"    {r.variant:12s} maxiter={r.maxiter:4d} "
                  f"ceiling={r.ceiling:.2f}x < measured={r.measured_speedup:.2f}x")
    else:
        print("Ceilings and measured speedups are mutually consistent at every maxiter.")
        print("The published 3.6x/21.8x pair was therefore a maxiter=30 artefact.")
else:
    print("No GPU -- consistency check needs both devices.")
    if len(split):
        display(split[["device", "variant", "maxiter", "torch_frac", "amdahl_ceiling"]])

## Part B -- does CUDA timing need an explicit sync?

Nothing in the estimator calls `torch.cuda.synchronize()`. CUDA kernels are asynchronous, so a
`perf_counter` around a torch callback can capture *launch* time instead of *execution* time,
which would understate `torch_frac` and overstate the ceiling.

There is a good reason to think it is fine anyway: every callback returns host arrays to CasADi,
and the device-to-host copy forces an implicit sync inside the timed region. But that is an
argument, not a measurement.

If `torch_frac` is materially higher with `sync=True`, the unsynchronised numbers -- including
the benchmark's published split -- are wrong for CUDA.

In [ ]:
if "cuda" in DEVICES:
    rows_b = []
    for exact in (False, True):
        for sync in (False, True):
            print(f"  cuda | {'exact' if exact else 'gauss-newton'} | sync={sync}",
                  flush=True)
            try:
                rows_b.append(profile_split("cuda", exact, maxiter=100, sync=sync))
            except Exception as exc:
                print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)
    syncdf = pd.DataFrame(rows_b)
    display(syncdf[["variant", "sync", "solve_s", "torch_frac", "amdahl_ceiling",
                    "hess_s", "gjac_s"]])
    for variant in ("gauss-newton", "exact"):
        a = syncdf[(syncdf.variant == variant) & (~syncdf.sync)]
        b = syncdf[(syncdf.variant == variant) & (syncdf.sync)]
        if len(a) and len(b):
            d = float(b.torch_frac.iloc[0]) - float(a.torch_frac.iloc[0])
            verdict = ("timing is VALID without sync" if abs(d) < 0.02
                       else "*** unsynchronised timing MISATTRIBUTES GPU work ***")
            print(f"  {variant:12s} torch_frac {float(a.torch_frac.iloc[0]):.3f} "
                  f"-> {float(b.torch_frac.iloc[0]):.3f} ({d:+.3f})  {verdict}")
else:
    print("No GPU -- this section is CUDA-specific.")

## Part C -- the Hessian chunk divisor

`_hchunk = _deriv_chunk // 2` halves the Hessian's vmap unconditionally.

On this problem the arithmetic is stark: `Da = 15`, 4 sensors, so each segment's second-order
buffer is ~0.82 MB and all 360 segments need **0.30 GB against a 2 GB budget**. `_deriv_chunk`
therefore saturates at `n_seg = 360` -- the Jacobian runs as one vmap -- while `_hchunk = 180`
splits the Hessian into **two sequential passes for no memory reason**.

`TWIN4BUILD_DERIV_BYTES` cannot fix this. `_deriv_chunk` is capped at the *segment count*, not
the budget, so no budget lifts `_hchunk` past `n_seg/2`. That is why the divisor itself is now
exposed as `TWIN4BUILD_HESS_CHUNK_DIV`.

Expect at most ~2x from removing it -- worth having, but far too small on its own to explain a
1.71x-against-21.8x gap. The sweep goes past 1 (values below 1 *enlarge* the chunk beyond the
Jacobian's) to find where memory actually starts to bind.

In [ ]:
DIVS = [2, 1]   # 2 = current default; 1 = no halving
rows_c = []
for dev in DEVICES:
    for div in DIVS:
        print(f"  {dev} | exact | HESS_CHUNK_DIV={div}", flush=True)
        try:
            r = profile_split(dev, True, maxiter=100, hess_div=div)
            rows_c.append(r)
        except RuntimeError as exc:
            # OOM is the expected failure when the chunk grows too large --
            # report it as a finding, not a crash.
            print(f"    OOM/RuntimeError at div={div}: {str(exc)[:120]}", flush=True)
            rows_c.append({"device": dev, "variant": "exact", "hess_div": div,
                           "solve_s": float("nan"), "error": str(exc)[:120]})
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)

chunk = pd.DataFrame(rows_c)
cols = [c for c in ["device", "hess_div", "solve_s", "hess_s", "torch_frac",
                    "amdahl_ceiling", "error"] if c in chunk.columns]
display(chunk[cols])

for dev in DEVICES:
    d2 = chunk[(chunk.device == dev) & (chunk.hess_div == 2)]
    d1 = chunk[(chunk.device == dev) & (chunk.hess_div == 1)]
    if len(d2) and len(d1) and np.isfinite(d1.solve_s.iloc[0]):
        s2, s1 = float(d2.solve_s.iloc[0]), float(d1.solve_s.iloc[0])
        h2 = float(d2.hess_s.iloc[0]); h1 = float(d1.hess_s.iloc[0])
        print(f"  {dev:5s} solve {s2:7.1f}s -> {s1:7.1f}s ({s2 / s1:.2f}x)   "
              f"hess {h2:7.1f}s -> {h1:7.1f}s ({h2 / h1:.2f}x)")

## Part D -- end to end with the best settings

Whatever Parts A-C show, the question that matters is whether the exact-Hessian GPU arm actually
gets faster. This runs the full arm at the benchmark's 300 iterations with the default divisor
and with the best one found above, and reports **fit as well as time** -- a runtime change that
moves the optimum is not a free win.

In [ ]:
SD = {"office_temperature_sensor": 0.05, "office_valve_position_sensor": 0.025,
      "office_damper_position_sensor": 0.025, "office_co2_sensor": 15.0}


def score(model):
    out, pooled = {}, 0.0
    for cid, sd in SD.items():
        c = model.components[cid]
        sim = c.output["measuredValue"].history()[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        act = c.time_series_input.values[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        r = float(np.sqrt(np.mean((sim - act) ** 2)))
        out[cid.replace("office_", "").replace("_sensor", "")] = r
        pooled += (r / sd) ** 2
    out["pooled"] = pooled
    return out


def _carry(entry, model):
    comps, attr, _x0, lo, hi = entry[:5]
    comp = comps[0] if isinstance(comps, list) else comps
    v = float(rgetattr(comp, attr).get().reshape(-1)[0])
    eps = 1e-9 * (hi - lo)
    return (comps, attr, min(max(v, lo + eps), hi - eps), lo, hi, *entry[5:])


def run_arm(device, hess_div=2, maxiter=300):
    prev = os.environ.get("TWIN4BUILD_HESS_CHUNK_DIV")
    os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = str(hess_div)
    try:
        model = build_model(device, tag="arm")
        sim = tb.Simulator(model)
        est = tb.Estimator(sim)
        params = build_parameters(model)
        meas = build_measurements(model)
        t0 = time.perf_counter()
        est.estimate(START, END, STEP, params, meas, n_warmup=N_WARMUP,
                     method=("scipy", "SLSQP", "ad"),
                     options={"maxiter": 5, "fast": True})
        p2 = [_carry(e, model) for e in params]
        opts = dict(COLLOC_OPTS)
        opts.update({"maxiter": maxiter, "exact_hessian": True})
        r = est.estimate(START, END, STEP, p2, meas, n_warmup=N_WARMUP,
                         method=("casadi", "ipopt", "ad", "collocation"), options=opts)
        wall = time.perf_counter() - t0
        seed = r.get("estimated_initial_state", {}) or {}
        model.set_save_simulation_result(flag=True)
        sim.simulate(step_size=STEP, start_time=START, end_time=END,
                     after_initialize=lambda: [model.get_component(c).set_state(x)
                                               for c, x in seed.items()])
        row = {"device": device, "hess_div": hess_div, "seconds": wall,
               "iterations": r["iterations"]}
        row.update(score(model))
        return row
    finally:
        if prev is None:
            os.environ.pop("TWIN4BUILD_HESS_CHUNK_DIV", None)
        else:
            os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = prev


rows_d = []
for dev in DEVICES:
    for div in DIVS:
        print(f"  {dev} | exact | div={div} | full arm", flush=True)
        try:
            rows_d.append(run_arm(dev, hess_div=div))
            print(f"    {rows_d[-1]['seconds']:.0f}s pooled={rows_d[-1]['pooled']:.2f}",
                  flush=True)
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)

arms = pd.DataFrame(rows_d)
display(arms)
print()
print(f"CPU: {HW['cpu']} ({HW['cores']} cores) | GPU: {HW['gpu']} | fp64 {HW['fp64']}")
print("Quote these WITH the hardware line above.")
print()
print("Reference (A100 run, gpu_benchmark_collocation.ipynb):")
print("  cpu collocation_exact 1339.1s pooled 30.78 | cuda collocation_exact 784.6s pooled 32.91")

## How to read the results

**Part A settles whether there is anything to explain.** If `torch_frac` rises with `maxiter`
and the ceilings become consistent with the measured speedups, then the published 3.6x/21.8x
pair was a `maxiter=30` artefact, and the exact Hessian's "shortfall" was measured against a
number that was never real. Check `setup_s`: that is the fixed cost being amortised.

**Part B is a validity check on every CUDA timing in this project**, including the benchmark's.
A large `torch_frac` shift under `sync=True` invalidates the unsynchronised split; a small one
confirms the implicit device-to-host sync was doing the job.

**Part C bounds the chunking fix.** Two sequential passes cannot cost more than ~2x, so even a
perfect result here does not close a 12x gap. It is worth having on its own merits -- splitting
a vmap that fits in memory is pure loss -- but do not expect it to be the answer.

**Part D is the only number that matters for users.** Watch `pooled` next to `seconds`: the
arms stop on a Gauss-Newton-style acceptable-level test, so a faster run that lands on a
different point of a flat ridge is not necessarily a better one.

**What would remain unexplained.** If Part A shows consistent ceilings, Part B shows valid
timing, and Part C gives its ~2x, and the exact Hessian is *still* far below its ceiling, then
the remaining candidates are: per-kernel occupancy (the `vmap(jacfwd(jacrev))` over 15 states
may simply be too small to fill an A100), fp64 tensor-core paths not being used by `matrix_exp`,
and the `n_links` vs `n_seg` split running two separate vmaps where one would do. Those need a
kernel-level profile (`torch.profiler` or Nsight), which is the honest next step rather than
more guessing at this level.

**Kernel-level profiling** now lives in its own notebook,
`gpu_profile_hessian_kernels.ipynb` -- it runs in ~5 minutes and does not
require re-running this grid.